**CALCULATING VaR AND CVaR ON 5-ASSET PORTFOLIO**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

**PREPARING DATA FROM YAHOO FINANCE**

In [2]:
tickers = ['GS','MSCI','V','CRISIL.NS','HDFCBANK.NS',]

In [4]:
data = yf.download(tickers=tickers,start='2019-01-01',end='2026-01-01')['Close']
print(data.head())

/tmp/ipykernel_1836/2049869090.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers=tickers,start='2019-01-01',end='2026-01-01')['Close']
[*********************100%***********************]  5 of 5 completed

Ticker        CRISIL.NS          GS  HDFCBANK.NS        MSCI           V
Date                                                                    
2019-01-01  1430.624146         NaN   496.185852         NaN         NaN
2019-01-02  1435.571289  144.753723   491.658447  135.801620  126.068077
2019-01-03  1447.055786  142.633270   487.800964  130.626129  121.525024
2019-01-04  1436.808228  147.294876   489.106049  135.487381  126.760460
2019-01-07  1448.645996  148.111084   489.845215  136.115845  129.046219


In [5]:
data.isnull().sum().sum()

np.int64(340)

In [7]:
data_clean = data.dropna()
data_clean

Ticker,CRISIL.NS,GS,HDFCBANK.NS,MSCI,V
Date,,,,,
2019-01-02,1435.571289,144.753723,491.658447,135.801620,126.068077
2019-01-03,1447.055786,142.633270,487.800964,130.626129,121.525024
2019-01-04,1436.808228,147.294876,489.106049,135.487381,126.760460
2019-01-07,1448.645996,148.111084,489.845215,136.115845,129.046219
2019-01-08,1438.222046,147.564102,485.768250,138.943893,129.748093
...,...,...,...,...,...
2025-12-24,4240.708496,902.036438,980.975220,577.381165,353.675964
2025-12-26,4209.180664,898.332336,975.958191,580.596802,353.536560
2025-12-29,4254.050781,883.614990,975.564697,581.361023,353.148132


In [8]:
data_clean.isnull().sum().sum()

np.int64(0)

In [15]:
returns = data_clean.pct_change(fill_method=None).dropna()
returns

Ticker,CRISIL.NS,GS,HDFCBANK.NS,MSCI,V
Date,,,,,
2019-01-03,0.008000,-0.014649,-0.007846,-0.038111,-0.036037
2019-01-04,-0.007082,0.032682,0.002675,0.037215,0.043081
2019-01-07,0.008239,0.005541,0.001511,0.004639,0.018032
2019-01-08,-0.007196,-0.003693,-0.008323,0.020777,0.005439
2019-01-09,0.005927,0.006273,0.006562,0.009778,0.011769
...,...,...,...,...,...
2025-12-24,0.004330,0.010059,0.000602,0.000774,0.004980
2025-12-26,-0.007435,-0.004106,-0.005114,0.005569,-0.000394
2025-12-29,0.010660,-0.016383,-0.000403,0.001316,-0.001099


In [16]:
returns.shape

(1673, 5)

In [18]:
cov_matrix = returns.cov()
cov_matrix

Ticker,CRISIL.NS,GS,HDFCBANK.NS,MSCI,V
Ticker,,,,,
CRISIL.NS,0.000452,0.000034,0.000056,0.000037,0.000027
GS,0.000034,0.000418,0.000070,0.000205,0.000193
HDFCBANK.NS,0.000056,0.000070,0.000257,0.000047,0.000060
MSCI,0.000037,0.000205,0.000047,0.000463,0.000204
V,0.000027,0.000193,0.000060,0.000204,0.000272


In [19]:
corr_matrix = returns.corr()
corr_matrix

Ticker,CRISIL.NS,GS,HDFCBANK.NS,MSCI,V
Ticker,,,,,
CRISIL.NS,1.000000,0.077564,0.165053,0.080848,0.078261
GS,0.077564,1.000000,0.213199,0.466452,0.574016
HDFCBANK.NS,0.165053,0.213199,1.000000,0.135153,0.226517
MSCI,0.080848,0.466452,0.135153,1.000000,0.575411
V,0.078261,0.574016,0.226517,0.575411,1.000000


In [20]:
mean_returns = returns.mean()
mean_returns

,0
Ticker,
CRISIL.NS,0.000875
GS,0.001281
HDFCBANK.NS,0.000538
MSCI,0.001089
V,0.000745


In [31]:
weights = np.array([0.2,0.2,0.2,0.2,0.2])

In [57]:
portfolio_returns = np.dot(weights,mean_returns)
print(f"The portfolio return is: {round(portfolio_returns,4)}")

The portfolio return is: 0.0009


In [42]:
portfolio_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
print(f"The portfolio variance is: {round(portfolio_variance,4)}")

The portfolio variance is: 0.0001


In [34]:
portoflio_standard_deviation = np.sqrt(portfolio_variance)
print(f"The portfolio standard deviation is: {round(portoflio_standard_deviation,4)}")

The portfolio standard deviation is: 0.0122


**USING SCIPY TO OPTIMIZE WEIGHTS**

In [25]:
from scipy.optimize import minimize

In [40]:
def calculate_portfolio_variance(weights, cov_matrix):
    return np.dot(weights.T, np.dot(cov_matrix, weights))

In [29]:
num_assets = 5
constraints = ({'type':'eq', 'fun':lambda w: np.sum(w) - 1})
bounds = tuple((0,1) for _ in range(num_assets))
initial_guess = num_assets * [1./num_assets]

In [44]:
result = minimize(calculate_portfolio_variance, initial_guess, args=(cov_matrix,),
                   method='SLSQP', bounds=bounds, constraints=constraints,
                   tol=1e-12)

In [48]:
min_var_weights = result.x
print(min_var_weights)

[0.21558422 0.07116901 0.36592469 0.0779096  0.26941247]


In [54]:
#Optimized portfolio returns
portfolio_returns = np.dot(min_var_weights,mean_returns)
print(f"The optimized portfolio return is: {round(portfolio_returns,4)}")

The optimized portfolio return is: 0.0008


In [58]:
#Optimized portfolio variance
portfolio_returns = np.dot(min_var_weights.T, np.dot(cov_matrix, min_var_weights))
print(f"The optimized portfolio variance is: {round(portfolio_returns,4)}")

The optimized portfolio variance is: 0.0001


In [59]:
#Optimized portfolio standard deviation
portfolio_returns = np.sqrt(portfolio_variance)
print(f"The optimized portfolio standard deviation is: {round(portfolio_returns,4)}")

The optimized portfolio standard deviation is: 0.0122
